# Learning recursions

This first exercise is a quick introduction to RNNs as models able to learn recursion patterns in sequential data.

In this exercise, we will use the same `torch` functions and framework as before

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt

We would like to see if a simple RNN can learn the recursion pattern of the famous Fibonacci sequence. 
We thus create below a dataset where inputs `X` are sub-sequences of the Fibonacci sequence while targets `y` are the next value of the sequence.

In [2]:
fibonacci = np.array([0,1,1,2,3,5,8,13,21,34,55,89,144,233,377,610,987])
d_inputs = 1 #input sesquence element dimensionality
n_steps = 2 #sequence length
n = len(fibonacci) - n_steps #number of sequences
X = np.zeros((n,n_steps,d_inputs),dtype=np.float32)
y = np.zeros((n,d_inputs),dtype=np.float32)
for i in range(n):
    X[i,:,0] = fibonacci[i:i+n_steps]
    y[i] = fibonacci[i+n_steps]

print(X.reshape((n,n_steps)))
print(y)

[[  0.   1.]
 [  1.   1.]
 [  1.   2.]
 [  2.   3.]
 [  3.   5.]
 [  5.   8.]
 [  8.  13.]
 [ 13.  21.]
 [ 21.  34.]
 [ 34.  55.]
 [ 55.  89.]
 [ 89. 144.]
 [144. 233.]
 [233. 377.]
 [377. 610.]]
[[  1.]
 [  2.]
 [  3.]
 [  5.]
 [  8.]
 [ 13.]
 [ 21.]
 [ 34.]
 [ 55.]
 [ 89.]
 [144.]
 [233.]
 [377.]
 [610.]
 [987.]]


As usual, we shall begin our computational graph by creating appropriate `tf.placeholder` instances.

**Q1** Create two `Tensor` instances with same shapes as `X` and `y`.

In [3]:
batch_size=n
Xt = torch.from_numpy(X).reshape(batch_size, n_steps, d_inputs)  
yt = torch.from_numpy(y).reshape(batch_size, d_inputs)   
print(Xt.shape,yt.shape)

torch.Size([15, 2, 1]) torch.Size([15, 1])


A sequence length of 4 has be chosen. We need to design a simple RNN architecture that has the capacity to learn that the target is equal to the sum of the last and penultimate elements of the input sequence.

We thus investigate a two-layer architecture:
* Layer 1 is a simple RNN layer with 2 neural units and thus will memorize a hidden state of length 2,
* Layer 2 is a fully connected layer that should learn to add.

Both layers will use linear activiations.

The code below creates `torch.Tensor` instances for the first layer. Remember that at $t=0$, Layer 1 must compute

$$ \mathbf{h}_0 = \mathbf{W}_x \cdot \mathbf{x}_0 + b. \text{ (1)}$$

Then, at every other time step, Layer 1 must compute

$$ \mathbf{h}_t = \mathbf{W}_x \cdot \mathbf{x}_t + \mathbf{W}_h \cdot \mathbf{h}_{t-1} + b. \text{ (2)}$$

The code below creates `torch.Tensor` instances for the trainable parameters of Layer 1

In [4]:
n_hs_neurons = 2 #number of neural units in the simple RNN cell

Wx = nn.Parameter(torch.randn(d_inputs, n_hs_neurons, dtype=torch.float32))
Wh = nn.Parameter(torch.randn(n_hs_neurons, n_hs_neurons, dtype=torch.float32))
b = nn.Parameter(torch.zeros(1, n_hs_neurons, dtype=torch.float32))

Now, to create the RNN layer, torch actully creates a computational graph corresponding to the unrolled version of the RNN layer. So, we need to separate our dataset w.r.t. time steps. This consists in slicing the input tensor w.r.t. the time dimension (axis 1 in our case). Then each portion of the input tensor is used separately in the graph !

The `torch.unbind` does this job:

In [5]:
X_seqs = torch.unbind(Xt,dim=1) 
X_seqs

(tensor([[  0.],
         [  1.],
         [  1.],
         [  2.],
         [  3.],
         [  5.],
         [  8.],
         [ 13.],
         [ 21.],
         [ 34.],
         [ 55.],
         [ 89.],
         [144.],
         [233.],
         [377.]]),
 tensor([[  1.],
         [  1.],
         [  2.],
         [  3.],
         [  5.],
         [  8.],
         [ 13.],
         [ 21.],
         [ 34.],
         [ 55.],
         [ 89.],
         [144.],
         [233.],
         [377.],
         [610.]]))

`X_seqs` is a list of tf tensors of shapes `(n,d_inputs)`, i.e. (13,1) in our case. For instance `X_seqs[0]` is a tensor that will contain each first sequence element of the training sequences, or in other words, the first column of the above printed `X`.

Now comes the computation part. We need to create the graph by first computing $\mathbf{h}_0$. Using `torch.matmul` this is quite simple. Then we need to loop on the remaining time steps, compute the current hidden state $\mathbf{h}_t$ using equation (2) and append this graph node to the list `hidden_states` (because the newly created node will be used in the next iteration).

**Q2** Fill the gaps in the code below to create the unrolled simple RNN layer.

In [12]:
h0 = torch.matmul(X_seqs[0], Wx)+ b
hidden_states = [h0]

print(hidden_states[-1].shape);

for i in range(1,n_steps):
    XWx=torch.matmul(X_seqs[i], Wx)
    HWh=torch.matmul(hidden_states[-1], Wh)
    hidden_states.append( XWx + HWh + b )

torch.Size([15, 2])


The last hidden state cannot be used as output $\hat{y}$ because it has size 2. So we need a second (fully connected layer) to compute a one-dimensional output from the last hidden state.

**Q3** Fill the gaps in the code below to create the fully connected layer that complete the network.

In [13]:
mlp = torch.nn.Linear(in_features= n_hs_neurons ,out_features=1)
output = mlp(hidden_states[-1])
mseloss = torch.nn.MSELoss()
print(yt.shape, output.shape)
loss = mseloss(yt, output)
print(loss)

torch.Size([15, 1]) torch.Size([15, 1])
tensor(173586.2812, grad_fn=<MseLossBackward0>)


Let's create a module regrouping all the components !

In [14]:
class SimpleRNN(torch.nn.Module):
    def __init__(self,n_hs_neurons,n_steps):
        super(SimpleRNN,self).__init__()
        
        self.n_hs_neurons = n_hs_neurons #number of neural units in the simple RNN cell
        self.n_steps = n_steps
        self.Wx = nn.Parameter(torch.randn(d_inputs, n_hs_neurons, dtype=torch.float32))
        self.Wh = nn.Parameter(torch.randn(n_hs_neurons, n_hs_neurons, dtype=torch.float32))
        self.b = nn.Parameter(torch.zeros(1, n_hs_neurons, dtype=torch.float32))

        self.mlp = torch.nn.Linear(in_features=n_hs_neurons,out_features=1)
        self.relu = torch.nn.ReLU()

    def __call__(self,x):
        X_seqs=torch.unbind(x,dim=1)
        h0 = torch.matmul(X_seqs[0], self.Wx)+ self.b
        hidden_states = [h0]
        for i in range(1,n_steps):
            XWx=torch.matmul(X_seqs[i], self.Wx)
            HWh=torch.matmul(hidden_states[-1], self.Wh)
            hidden_states.append( XWx + HWh + self.b)
        return self.mlp(hidden_states[-1])

We have also added a loss to the graph. We can now train the model and check if it works.

In [19]:
learning_rate = 0.001
epochs = 2000

mseloss = torch.nn.MSELoss()
mlp = torch.nn.Linear(in_features=n_hs_neurons,out_features=1)
optimizer = torch.optim.Adam([Wx,Wh,b,mlp.weight,mlp.bias], lr=learning_rate)

for epoch in range(epochs):
    optimizer.zero_grad()
    h0 = torch.matmul(X_seqs[0],Wx)+b
    hidden_states = [h0]
    #print(hidden_states[-1].shape);
    for i in range(1,n_steps):
        XWx=torch.matmul(X_seqs[i], Wx)
        HWh=torch.matmul(hidden_states[-1], Wh)
        hidden_states.append( XWx + HWh + b )
    output = mlp(hidden_states[-1])
    loss = mseloss(yt, output)
    if epoch % 20 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")
    loss.backward()
    optimizer.step()
    

Epoch 0 - Loss: 41378.9219
Epoch 20 - Loss: 29869.2930
Epoch 40 - Loss: 20696.7227
Epoch 60 - Loss: 13786.8037
Epoch 80 - Loss: 8804.7607
Epoch 100 - Loss: 5364.9219
Epoch 120 - Loss: 3102.7153
Epoch 140 - Loss: 1695.1282
Epoch 160 - Loss: 871.5997
Epoch 180 - Loss: 420.7286
Epoch 200 - Loss: 190.4623
Epoch 220 - Loss: 80.9519
Epoch 240 - Loss: 32.4874
Epoch 260 - Loss: 12.5298
Epoch 280 - Loss: 4.8819
Epoch 300 - Loss: 2.1563
Epoch 320 - Loss: 1.2530
Epoch 340 - Loss: 0.9749
Epoch 360 - Loss: 0.8948
Epoch 380 - Loss: 0.8726
Epoch 400 - Loss: 0.8659
Epoch 420 - Loss: 0.8631
Epoch 440 - Loss: 0.8610
Epoch 460 - Loss: 0.8590
Epoch 480 - Loss: 0.8570
Epoch 500 - Loss: 0.8550
Epoch 520 - Loss: 0.8529
Epoch 540 - Loss: 0.8508
Epoch 560 - Loss: 0.8486
Epoch 580 - Loss: 0.8464
Epoch 600 - Loss: 0.8442
Epoch 620 - Loss: 0.8419
Epoch 640 - Loss: 0.8395
Epoch 660 - Loss: 0.8371
Epoch 680 - Loss: 0.8347
Epoch 700 - Loss: 0.8322
Epoch 720 - Loss: 0.8297
Epoch 740 - Loss: 0.8272
Epoch 760 - Loss: 0

<div style="background-color: #f0fff0; border:2px solid #ffb703; border-radius:8px; padding:10px; color:black;">

Over epochs, the MSE Loss keeps decreasing and converges to a certain level showing that the model learns well.

In [21]:
learning_rate = 0.001
epochs = 2000

mseloss = torch.nn.MSELoss()
model = SimpleRNN(n_hs_neurons,n_steps)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
model.train()

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(Xt)
    loss = mseloss(yt, output)
    loss.backward()
    optimizer.step()

    if epoch%400==399:
        model.eval()
        print(yt.flatten().detach()-output.flatten().detach())
        print((yt.flatten().detach()-output.flatten().detach())/yt.flatten().detach())
        print(f"Epoch {epoch} | Loss {loss:.4f} | \r")
    

tensor([ -0.3403,   0.4782,   0.4178,   1.1759,   1.8736,   3.3294,   5.4829,
          9.0921,  14.8549,  24.2269,  39.3617,  63.8685, 103.5100, 167.6585,
        271.4484])
tensor([-0.3403,  0.2391,  0.1393,  0.2352,  0.2342,  0.2561,  0.2611,  0.2674,
         0.2701,  0.2722,  0.2733,  0.2741,  0.2746,  0.2748,  0.2750])
Epoch 399 | Loss 7938.2197 | 
tensor([-0.8071, -0.2702, -0.5985, -0.3898, -0.5094, -0.4204, -0.4510, -0.3925,
        -0.3647, -0.2783, -0.1641,  0.0364,  0.3511,  0.8664,  1.6964])
tensor([-8.0710e-01, -1.3511e-01, -1.9949e-01, -7.7965e-02, -6.3679e-02,
        -3.2339e-02, -2.1475e-02, -1.1545e-02, -6.6302e-03, -3.1273e-03,
        -1.1399e-03,  1.5612e-04,  9.3123e-04,  1.4203e-03,  1.7187e-03])
Epoch 799 | Loss 0.4012 | 
tensor([-0.8023, -0.2670, -0.5965, -0.3908, -0.5147, -0.4328, -0.4747, -0.4348,
        -0.4369, -0.3990, -0.3632, -0.2896, -0.1801,  0.0031,  0.2956])
tensor([-8.0226e-01, -1.3348e-01, -1.9884e-01, -7.8161e-02, -6.4331e-02,
        -3.3290e-02

<div style="background-color: #f0fff0; border:2px solid #ffb703; border-radius:8px; padding:10px; color:black;">

Same as before, the training error and the relative error decreased as expected over epochs which shows that the model learns.

In [22]:
print(model.Wh, model.Wx, model.b)

Parameter containing:
tensor([[ 0.2353, -0.2925],
        [ 1.0967,  0.0301]], requires_grad=True) Parameter containing:
tensor([[3.2797, 0.6864]], requires_grad=True) Parameter containing:
tensor([[0.2098, 0.4216]], requires_grad=True)


The next Fibonacci sequence elements are : \[1597,2584,4181,6765,10946,17711\]

**Q4** Check that the RNN model is not overfitted.

In [23]:
fibonacci_test = np.array([377,610,987,1597,2584,4181,6765,10946,17711])
n_test = len(fibonacci_test) - n_steps #number of sequences
X_test = np.zeros((n_test,n_steps,d_inputs),dtype=np.float32)
y_test = np.zeros((n_test,d_inputs),dtype=np.float32)
for i in range(n_test):
    X_test[i,:,0] = fibonacci_test[i:i+n_steps]
    y_test[i] = fibonacci_test[i+n_steps]

batch_size=n_test
Xt_test = torch.from_numpy(X_test).reshape(batch_size, n_steps, d_inputs)  
yt_test = torch.from_numpy(y_test).reshape(batch_size, d_inputs)   

output_test=model(Xt_test)
print(output_test-yt_test)
print((output_test-yt_test)/yt_test)

tensor([[ -0.2809],
        [ -0.7340],
        [ -1.4658],
        [ -2.6509],
        [ -4.5684],
        [ -7.6699],
        [-12.6895]], grad_fn=<SubBackward0>)
tensor([[-0.0003],
        [-0.0005],
        [-0.0006],
        [-0.0006],
        [-0.0007],
        [-0.0007],
        [-0.0007]], grad_fn=<DivBackward0>)


<div style="background-color: #f0fff0; border:2px solid #ffb703; border-radius:8px; padding:10px; color:black;">

As we can observe, the relative error test is actually low around 0.06%. Then, it shows that the model didn't overfit even if we add more elements to the sequences.